In [1]:
# @title 1. 安裝 RFdiffusion All Atom (極速版)
import os
import sys
import time

start_time = time.time()
print("🚀 啟動 RFdiffusionAA 安裝程序...")

# 1. 安裝與設定基礎依賴
print("⚙️ (1/5) 正在設定基礎環境 (Torch & Apptainer)...")

# 靜默安裝 PyTorch (這步最久，使用 -q 隱藏輸出)
!pip install -q torch==2.5.0
!pip uninstall -y -q torchvision torchaudio

# 安裝 aria2c 加速神器
if not os.path.exists("/usr/bin/aria2c"):
    !apt-get update -y > /dev/null
    !apt-get install -y aria2 > /dev/null

# 2. 下載程式碼 (使用 depth=1 只抓最新版)
if not os.path.isdir("rf_diffusion_all_atom"):
    print("📂 (2/5) 下載主程式碼...")
    !git clone --depth 1 https://github.com/baker-laboratory/rf_diffusion_all_atom.git
    os.chdir("rf_diffusion_all_atom")
else:
    os.chdir("rf_diffusion_all_atom")

# 3. 安裝 Apptainer (容器執行工具)
if not os.path.exists("/usr/bin/apptainer"):
    print("📦 (3/5) 安裝容器執行工具...")
    os.environ["APPTAINER_BINDPATH"] = "/content"
    !curl -J -O -s https://raw.githubusercontent.com/NeuroDesk/neurocommand/main/googlecolab_setup.sh
    !chmod +x googlecolab_setup.sh
    !./googlecolab_setup.sh > /dev/null 2>&1 # 隱藏雜訊

# 4. 極速下載模型與容器 (核心優化)
print("⬇️ (4/5) 正在全速下載模型與容器 (這會比平常快很多)...")

downloads = {
    "rf_se3_diffusion.sif": "http://files.ipd.uw.edu/pub/RF-All-Atom/containers/rf_se3_diffusion.sif",
    "RFDiffusionAA_paper_weights.pt": "http://files.ipd.uw.edu/pub/RF-All-Atom/weights/RFDiffusionAA_paper_weights.pt"
}

for filename, url in downloads.items():
    if not os.path.exists(filename):
        print(f"   正在下載: {filename} ...")
        # -x 16 代表開 16 條線程同時抓
        !aria2c -x 16 -s 16 -k 1M -o {filename} {url} > /dev/null 2>&1
        print(f"   ✅ {filename} 完成")
    else:
        print(f"   ✅ {filename} 已存在，跳過")

# 5. 初始化 Submodules
print("🔧 (5/5) 最後設定...")
!git submodule init > /dev/null
!git submodule update > /dev/null

end = int(time.time() - start_time)
print(f"\n🎉 RFdiffusionAA 安裝完成！總耗時: {end} 秒")

🚀 啟動 RFdiffusionAA 安裝程序...
⚙️ (1/5) 正在設定基礎環境 (Torch & Apptainer)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 

In [2]:
# @title RFdiffusion 萬能通用版 (自動適應有/無配體 + 完美微調破解)
# @markdown 🔴 **只需輸入 PDB ID，有配體自動抓取，無配體自動切換純蛋白模式並略過報錯**

import time
import os
import sys
import urllib.request
import re

# ==========================================
#        第一步：環境與變數設定 & 完美破解
# ==========================================
print("🔍 正在初始化環境...")

work_dir = "/content/rf_diffusion_all_atom"
if os.getcwd() != work_dir:
    os.makedirs(work_dir, exist_ok=True)
    os.chdir(work_dir)

sif_filename = "rf_se3_diffusion.sif"
if not os.path.exists(sif_filename):
    print(f"❌ 找不到映像檔，正在下載 (約 2GB)...")
    !wget https://files.ipd.uw.edu/pub/rf_diffusion/rf_se3_diffusion.sif

# ⚡ 安全還原並破解 run_inference.py
file_path = "run_inference.py"
if os.path.exists(file_path):
    # 強制還原剛剛被弄壞的檔案 (恢復原廠設定)
    os.system(f"git checkout {file_path} 2>/dev/null")

    with open(file_path, "r") as f:
        code = f.read()

    # 動態抓取檔案原有的縮排空白數量，保證 Python 語法絕對正確
    match = re.search(r'^(\s*)idealize_backbone\.rewrite\(out_unidealized, out_idealized\)', code, re.MULTILINE)

    if match:
        indent = match.group(1)
        safe_new_code = (
            f"{indent}try:\n"
            f"{indent}    idealize_backbone.rewrite(out_unidealized, out_idealized)\n"
            f"{indent}except Exception as e:\n"
            f"{indent}    print('\\n⚠️ [破解成功] 略過骨架微調，強制保留原始結果')\n"
            f"{indent}    import shutil\n"
            f"{indent}    shutil.copy(out_unidealized, out_idealized)"
        )
        code = code.replace(match.group(0), safe_new_code)

        with open(file_path, "w") as f:
            f.write(code)
        print("✅ 完美破解微調機制！縮排已自動對齊。")
    else:
        print("⚡ 程式碼已是最新狀態，無需重複破解。")

# @markdown **1. 輸入 PDB ID**
pdb_id = "1MBN" # @param {type:"string"}
output_prefix = "output/universal/sample" # @param {type:"string"}
contig_settings = "['150-150']" # @param {type:"string"}

# @markdown **2. 擴散模型參數**
diffuser_T = 30 # @param {type:"slider", min:10, max:100, step:5}
num_designs = 1 # @param {type:"integer"}

# 自動下載邏輯
pdb_id = pdb_id.strip().lower()
input_dir = "input"
os.makedirs(input_dir, exist_ok=True)
input_pdb = os.path.join(input_dir, f"{pdb_id}.pdb")
cleaned_pdb = os.path.join(input_dir, f"{pdb_id}_cleaned.pdb")

if not os.path.exists(input_pdb):
    print(f"\n📥 正在從 RCSB PDB 下載 {pdb_id.upper()}...")
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb", input_pdb)

# ==========================================
#        第二步：自動解析配體 (智慧判斷)
# ==========================================
print("\n🧬 正在分析 PDB 配體...")
ignore_list = {"HOH", "WAT", "SO4", "PO4", "GOL", "EDO", "ACT", "NAG", "CL", "NA", "K", "MG", "CA", "ZN", "FMT"}
found_ligands = set()

with open(input_pdb, 'r') as f:
    for line in f:
        if line.startswith("HETATM"):
            res_name = line[17:20].strip()
            if res_name not in ignore_list:
                found_ligands.add(res_name)

has_ligand = len(found_ligands) > 0
ligand_code = None

if has_ligand:
    ligand_code = list(found_ligands)[0]
    print(f"🎯 [模式切換] 偵測到配體，啟動【配體結合設計模式】(鎖定目標: {ligand_code})")
else:
    print(f"ℹ️ [模式切換] 未偵測到配體，啟動【純蛋白質擴散模式】(已啟用破解防護)")

# ==========================================
#        第三步：嚴格清洗 PDB (骨架完整性檢查)
# ==========================================
print("\n🧹 正在清洗 PDB (剔除殘缺骨架與雜質)...")

standard_aas = {"ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
                "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL"}

valid_residues = set()
with open(input_pdb, 'r') as f_in:
    for line in f_in:
        if line.startswith("ATOM"):
            res_name = line[17:20].strip()
            atom_name = line[12:16].strip()
            chain = line[21]
            res_seq = line[22:26]
            if res_name in standard_aas and atom_name == "CA":
                valid_residues.add((chain, res_seq))

with open(input_pdb, 'r') as f_in, open(cleaned_pdb, 'w') as f_out:
    for line in f_in:
        if line.startswith("ATOM"):
            chain = line[21]
            res_seq = line[22:26]
            if (chain, res_seq) in valid_residues:
                alt_loc = line[16]
                if alt_loc in [' ', 'A']:
                    clean_line = line[:16] + ' ' + line[17:26] + ' ' + line[27:]
                    f_out.write(clean_line)

        elif line.startswith("HETATM") and has_ligand:
            res_name = line[17:20].strip()
            if res_name == ligand_code:
                alt_loc = line[16]
                if alt_loc in [' ', 'A']:
                    clean_line = line[:16] + ' ' + line[17:26] + ' ' + line[27:]
                    f_out.write(clean_line)

print(f"✅ 清洗完成！輸出至: {cleaned_pdb}")

# ==========================================
#        第四步：動態執行 RFdiffusion
# ==========================================
os.makedirs(os.path.dirname(output_prefix), exist_ok=True)
print(f"\n🚀 開始執行 RFdiffusion...")

abs_cleaned_pdb = os.path.abspath(cleaned_pdb)

if has_ligand:
    !singularity run --nv {sif_filename} -u run_inference.py \
      inference.deterministic=True \
      diffuser.T={diffuser_T} \
      inference.output_prefix={output_prefix} \
      inference.input_pdb="{abs_cleaned_pdb}" \
      contigmap.contigs="{contig_settings}" \
      inference.ligand={ligand_code} \
      inference.num_designs={num_designs} \
      inference.design_startnum=0
else:
    !singularity run --nv {sif_filename} -u run_inference.py \
      inference.deterministic=True \
      diffuser.T={diffuser_T} \
      inference.output_prefix={output_prefix} \
      inference.input_pdb="{abs_cleaned_pdb}" \
      contigmap.contigs="{contig_settings}" \
      inference.num_designs={num_designs} \
      inference.design_startnum=0

if os.path.exists(f"{output_prefix}_0.pdb"):
    print(f"\n🎉 執行成功！已生成檔案: {output_prefix}_0.pdb")
else:
    print(f"\n❌ 執行結束，但未生成檔案，請查看上方日誌。")

🔍 正在初始化環境...
✅ 完美破解微調機制！縮排已自動對齊。

📥 正在從 RCSB PDB 下載 1MBN...

🧬 正在分析 PDB 配體...
🎯 [模式切換] 偵測到配體，啟動【配體結合設計模式】(鎖定目標: HEM)

🧹 正在清洗 PDB (剔除殘缺骨架與雜質)...
✅ 清洗完成！輸出至: input/1mbn_cleaned.pdb

🚀 開始執行 RFdiffusion...
/bin/sh: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
DGL backend not selected or invalid.  Assuming PyTorch for now.
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
[05:34:47] /opt/dgl/src/runtime/tensordispatch.cc:43: TensorDispatcher: dlopen failed: /opt/conda/envs/env/lib/python3.9/site-packages/dgl/tensoradapter/pytorch/libtensoradapter_pytorch_1.13.1.so: cannot open shared object file: No such file or directory
Using backend: pytorch
/opt/conda/envs/env/lib/python3.9/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'aa': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1

In [3]:
from google.colab import files

files.download('/content/rf_diffusion_all_atom/output/universal/sample_0.pdb')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title 2. LigandMPNN 環境準備 (自動重啟)
import os
import sys
import time

install_flag_path = '/content/install_done_flag'

if os.path.exists(install_flag_path):
    print("✅ 環境已準備就緒！請直接執行下一個步驟 (Step 3)。")

else:
    print("🚀 正在優化 Python 運算環境 (約需 1 分鐘)...")

    # 靜默安裝，不顯示紅字警告
    os.system('pip install -q --no-warn-conflicts numpy pyparsing torch')

    # 建立完成標記
    os.system(f'touch {install_flag_path}')

    print("--------------------------------------------------")
    print("⚠️ 系統即將【自動重啟】以套用設定。")
    print("   若看到『工作階段已當機』或『連線中斷』是正常的！")
    print("--------------------------------------------------")
    print("👉 重啟後，請直接執行【第 3 步驟】(不需要再按一次這個按鈕)。")

    time.sleep(3)
    os.kill(os.getpid(), 9)

🚀 正在優化 Python 運算環境 (約需 1 分鐘)...
--------------------------------------------------
⚠️ 系統即將【自動重啟】以套用設定。
   若看到『工作階段已當機』或『連線中斷』是正常的！
--------------------------------------------------
👉 重啟後，請直接執行【第 3 步驟】(不需要再按一次這個按鈕)。


In [ ]:
# @title 3. 安裝 LigandMPNN 主程式
import os
import sys
import time

# 標記檔名
INSTALL_FLAG = "MPNN_READY_FLAG"

# 檢查是否已經安裝好且能用
try:
    import prody
    if os.path.exists(INSTALL_FLAG):
        print("✅ LigandMPNN 環境已就緒！請繼續往下執行。")
        sys.path.append(os.path.abspath("/content/LigandMPNN"))
        # 提前結束，不執行後面安裝
        sys.exit()
except ImportError:
    pass # 繼續執行安裝

print("🚀 開始安裝 LigandMPNN (不會太久)...")

# 1. 下載程式碼 (靜默)
if not os.path.exists("LigandMPNN"):
    print("   ⬇️ 下載程式庫與參數...")
    !git clone https://github.com/ullahsamee/LigandMPNN.git > /dev/null 2>&1

    # 進入目錄下載模型參數
    os.chdir("LigandMPNN")
    if not os.path.exists("model_params"):
        !bash get_model_params.sh "./model_params" > /dev/null 2>&1
    os.chdir("..")

# 2. 安裝依賴 (靜默)
print("   📦 安裝生物資訊套件...")
!pip install -q --upgrade setuptools wheel
!pip install -q prody pyparsing biopython

# 3. 設定完成標記
with open(INSTALL_FLAG, 'w') as f:
    f.write("Done")

# 加入路徑
sys.path.append(os.path.abspath("/content/LigandMPNN"))

print("✅ 安裝完成！")

# 最終確認
try:
    import prody
    print(f"🎯 驗證成功 (ProDy v{prody.__version__})")
except ImportError:
    print("⚠️ 驗證失敗，請嘗試手動重啟執行階段 (Runtime -> Restart Session)。")

🚀 開始安裝 LigandMPNN (不會太久)...
   ⬇️ 下載程式庫與參數...
   📦 安裝生物資訊套件...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.25 requires torchvision, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
✅ 安裝完成！
🎯 驗證成功 (ProDy v2.6.1)


修補沒有ligand的蛋白質

In [ ]:
# 使用 pip 安裝 openmm 與 pdbfixer
!pip install openmm pdbfixer

# 再次執行 pdbfixer 來補齊缺失的重原子
!pdbfixer /content/rf_diffusion_all_atom/output/universal/sample_0.pdb --output=/content/sample_0_fixed.pdb --add-atoms=heavy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 12.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 67.0 MB/s eta 0:00:00
  Created wheel for pdbfixer: filename=pdbfixer-1.12.0-py3-none-any.whl size=681766 sha256=9da51f817787e5e3be9cb0b592222c0bc1dc325559785c891ecb48ec686fa8f9
  Stored in directory: /root/.cache/pip/wheels/79/ec/63/2ad240b7fac835f490bd1ecb748da87c99ae65618770c73a98
Successfully built pdbfixer


In [ ]:
%cd /content/LigandMPNN
!python run.py \
        --seed 111 \
        --pdb_path "/content/sample_0_fixed.pdb" \
        --out_folder "./outputs/default" \
        --number_of_batches 1 \
        --batch_size 10

/content/LigandMPNN
/content/LigandMPNN/run.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device)
Designing prot

In [ ]:
# @title 4. 安裝 PDBFixer (結構修復工具)
import os
import sys

print("🛠️ 正在安裝 OpenMM 與 PDBFixer...")

# 1. 安裝 OpenMM (靜默)
!pip install -q openmm

# 2. 從 GitHub 直接安裝 PDBFixer (靜默)
!pip install -q git+https://github.com/openmm/pdbfixer.git

# 3. 驗證安裝
try:
    from pdbfixer import PDBFixer
    from openmm.app import PDBFile
    print("✅ PDBFixer 安裝成功！所有環境準備完畢。")
except ImportError:
    print("❌ 安裝失敗，請檢查網路或稍後重試。")
input_dir = "./outputs/default/backbones"
output_dir = "./outputs/with_sidechains"
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(input_dir):
    print(f"⚠️ 找不到輸入資料夾: {input_dir}")
else:
    files = [f for f in os.listdir(input_dir) if f.endswith(".pdb")]
    print(f"📂 準備處理 {len(files)} 個檔案...")

    for pdb_file in files:
        try:
            full_input = os.path.join(input_dir, pdb_file)
            full_output = os.path.join(output_dir, pdb_file)

            print(f"   -> 正在修復: {pdb_file} ...", end="")

            fixer = PDBFixer(filename=full_input)
            fixer.findMissingResidues()
            fixer.findMissingAtoms() # 長出側鏈
            fixer.addMissingAtoms()

            with open(full_output, 'w') as f:
                PDBFile.writeFile(fixer.topology, fixer.positions, f)
            print(" 完成")

        except Exception as e:
            print(f"\n   ❌ 失敗 ({pdb_file}): {e}")

    print(f"\n✨ 全部完成！檔案在: {output_dir}")
import os

# --- 設定路徑 ---
# 1. 剛剛長好側鏈的蛋白 (只有蛋白，沒有 Ligand)
protein_dir = "./outputs/with_sidechains"
# 2. LigandMPNN 原始輸出的骨架檔 (只有主鏈，但有 Ligand!)
original_dir = "./outputs/default/backbones"
# 3. 最終輸出的資料夾
final_output_dir = "./outputs/final_complete"

os.makedirs(final_output_dir, exist_ok=True)

# 檢查資料夾
if not os.path.exists(protein_dir):
    print(f"❌ 找不到側鏈資料夾: {protein_dir}")
else:
    files = [f for f in os.listdir(protein_dir) if f.endswith(".pdb")]
    print(f"🔧 開始合併 Ligand，共 {len(files)} 個檔案...")

    for pdb_file in files:
        protein_path = os.path.join(protein_dir, pdb_file)
        original_path = os.path.join(original_dir, pdb_file)
        final_path = os.path.join(final_output_dir, pdb_file)

        # 1. 讀取有側鏈的蛋白 (只要 ATOM 行)
        protein_lines = []
        with open(protein_path, 'r') as f:
            for line in f:
                # 保留晶胞參數(CRYST1)和原子(ATOM)，過濾掉結束符號(END)
                if line.startswith("ATOM") or line.startswith("CRYST1"):
                    protein_lines.append(line)

        # 2. 讀取原始檔案中的 Ligand (只要 HETATM 行)
        ligand_lines = []
        if os.path.exists(original_path):
            with open(original_path, 'r') as f:
                for line in f:
                    # 通常 Ligand 是 HETATM 開頭
                    if line.startswith("HETATM") or line.startswith("CONECT"):
                        ligand_lines.append(line)
        else:
            print(f"⚠️ 警告: 找不到原始檔案 {original_path}，無法提取 Ligand")

        # 3. 合併寫入新檔案
        with open(final_path, 'w') as f:
            # 先寫蛋白
            f.writelines(protein_lines)
            # 再寫 Ligand
            f.writelines(ligand_lines)
            # 最後補上 END
            f.write("END\n")

        print(f"   -> {pdb_file} 合體成功！ (蛋白+側鏈+Ligand)")

    print(f"\n🎉 完美！最終檔案都在: {final_output_dir}")

# --- 自動打包下載 ---
import shutil
from google.colab import files

print("📦 正在打包下載...")
shutil.make_archive("Final_Complexes", 'zip', final_output_dir)
files.download("Final_Complexes.zip")

🛠️ 正在安裝 OpenMM 與 PDBFixer...
  Preparing metadata (setup.py) ... done
✅ PDBFixer 安裝成功！所有環境準備完畢。
📂 準備處理 11 個檔案...
   -> 正在修復: sample_0_fixed_9.pdb ... 完成
   -> 正在修復: sample_0_fixed_7.pdb ... 完成
   -> 正在修復: sample_0_fixed_8.pdb ... 完成
   -> 正在修復: sample_0_fixed_5.pdb ... 完成
   -> 正在修復: sample_0_fixed_1.pdb ... 完成
   -> 正在修復: sample_0_fixed_10.pdb ... 完成
   -> 正在修復: sample_0_fixed_4.pdb ... 完成
   -> 正在修復: sample_0_fixed_6.pdb ... 完成
   -> 正在修復: sample_0_fixed_2.pdb ... 完成
   -> 正在修復: 1BC8_1.pdb ... 完成
   -> 正在修復: sample_0_fixed_3.pdb ... 完成

✨ 全部完成！檔案在: ./outputs/with_sidechains
🔧 開始合併 Ligand，共 11 個檔案...
   -> sample_0_fixed_9.pdb 合體成功！ (蛋白+側鏈+Ligand)
   -> sample_0_fixed_7.pdb 合體成功！ (蛋白+側鏈+Ligand)
   -> sample_0_fixed_8.pdb 合體成功！ (蛋白+側鏈+Ligand)
   -> sample_0_fixed_5.pdb 合體成功！ (蛋白+側鏈+Ligand)
   -> sample_0_fixed_1.pdb 合體成功！ (蛋白+側鏈+Ligand)
   -> sample_0_fixed_10.pdb 合體成功！ (蛋白+側鏈+Ligand)
   -> sample_0_fixed_4.pdb 合體成功！ (蛋白+側鏈+Ligand)
   -> sample_0_fixed_6.pdb 合體成功！ (蛋白+側鏈+Ligand)
   -> 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 前往ColabFold  網址 https://colab.research.google.com/drive/1Xbo_8ozPn6axIXAb_f6tWyjwmKJWS_WI?usp=sharing